## 🎬 진행자용 Live Demo 체크리스트 (Supabase 처음 설치부터)

> 강의 중 순서를 까먹지 않으려는 메모. 학생도 같은 순서로 따라온다.

**1. Supabase 프로젝트 만들기** — 대시보드에서 학생 각자 1개 생성.

**2. 키 세 개를 헷갈리지 말 것**

| 키 | 환경변수 이름 | 쓰는 곳 | 성격 |
|---|---|---|---|
| Access Token | `SUPABASE_ACCESS_TOKEN` | Claude Code `.mcp.json`에서만 | 계정 전체 권한 — 노트북 코드에는 절대 안 씀 |
| Publishable Key | `SUPABASE_PUBLISHABLE_KEY` (= anon key) | 이 노트북의 SDK 코드 | 프로젝트 한정, RLS가 방어선 |
| ~~Secret Key~~ (= service_role) | 안 씀 | — | RLS를 전부 무시하는 관리자 키 — 강의에서 사용 금지 |

**3. Supabase MCP 설치** — [§610 노트](../notes/FD-04-610__supabase-mcp-natural-language.md) 참고. `.mcp.json`에 project-ref + Access Token 등록 → Claude Code 재시작 → `list_tables`로 연결 확인.

**4. 표 만들면 바로 RLS 끄기** — `create table` 다음 줄에서 바로 `alter table ... disable row level security`까지 실행한다. 빼먹으면 SDK에서 `42501 new row violates row-level security policy` 에러가 난다.

**5. 수업 끝나면 초기화** — `.mcp.json` 삭제하고, 발급했던 Access Token은 Supabase 대시보드에서 폐기(revoke)한다. 다음 기수는 새로 발급.


# ☁️ OpenDART 응답을 Supabase로 바로 적재 — Student

OpenDART API에서 삼성전자 2025년 연결 재무제표를 직접 받고, SQLite나 CSV를 거치지 않고 Supabase에 저장한다.

> **🧭 이 실습 순서**
> 1. [§300](../notes/FD-04-300__opendart-hard-way.md)에서 OpenDART 키와 API 응답을 준비한다.
> 2. [§410](../notes/FD-04-410__mcp-agentic-install.md)에서 OpenDART MCP를 설치·연결한다.
> 3. [§610](../notes/FD-04-610__supabase-mcp-natural-language.md)에서 개인 Supabase 프로젝트를 만들고, 자연어 요청으로 표를 만들고 INSERT·SELECT한다.
> 4. 준비를 마치면 이 노트북의 **셀 1**로 돌아와 같은 자료를 [§600 SDK 코드](../notes/FD-04-600__cloud-supabase.md)로 적재하고 다시 조회한다.
>    SDK로 같은 행을 다시 읽어 앞서 확인한 결과와 비교하는 단계다. MCP·SQL과 SDK의 방법을 비교하고, 조회 결과가 예상 행과 맞는지 확인한다.
>
> **✅ 시작 전 완료 확인**
> - [ ] §300: `OPENDART_API_KEY`가 준비됐고 실제 OpenDART 응답 `[live]`와 결과를 확인했다. 키 값은 화면에 표시하지 않는다.
> - [ ] §410: `opendart-mcp` 도구가 목록에 보이고 테스트 조회를 마쳤다.
> - [ ] §610: OAuth가 연결됐고 `CREATE`·`INSERT`·`SELECT`로 예상 행을 확인했다.
> - §300 결과가 `[fixture]`뿐이면 오프라인 수업용 대체 자료이므로 §610으로 진행하거나 클라우드 E2E 경로를 완료했다고 표시하지 않는다. 먼저 [SQLite 오프라인 실습](FD-04-800__mcp-load-and-measure-student.ipynb)을 마치고, 실제 `[live]` 응답을 받은 뒤 §610으로 돌아와 이 노트북 **셀 1**부터 진행한다. `[fixture]`를 실제 OpenDART 결과처럼 Supabase에 넣지 않는다.
> - 앞 단계에서 확인한 종목·연도·네 계정 값과 조회 결과처럼 비밀이 아닌 사실만 기억한다. 키·secret은 가져오지 않는다. SDK에서 같은 `[live]` 행을 다시 적재·조회해 MCP·SQL 결과와 비교한다. 완료했으면 이 노트북의 **셀 1**로 돌아온다.
>

> **🖥️ 환경 설정**
>
> - VS Code에서 `FD_26/.venv` 커널을 선택한다. 모든 실행과 설치는 FD_26에서 한다.
> - `requests`·`python-dotenv`·`supabase`가 없으면 FD_26 루트에서 `uv add requests python-dotenv supabase`를 실행한다.
> - FD_26 루트의 환경변수 파일에 `OPENDART_API_KEY`, `SUPABASE_URL`, `SUPABASE_PUBLISHABLE_KEY`가 있어야 한다. 키 값은 코드·출력에 쓰지 않는다.
> - 개인 Supabase 프로젝트와 `financials` 표는 위 순서의 §610에서 준비한다.
> - Supabase를 아직 준비하지 못했다면 이 노트북을 건너뛰고 SQLite 오프라인 실습을 진행한다.

| 단축키 | 동작 |
|---|---|
| `Shift + Enter` | 셀 실행 후 다음 셀 이동 |
| `Esc → A` | 위에 셀 삽입 |
| `Esc → B` | 아래에 셀 삽입 |
| `Esc → DD` | 셀 삭제 |


In [ ]:
# 🖥️ 환경 설정
# - VS Code에서 FD_26/.venv 커널을 선택했는지 먼저 확인한다 (실행·설치는 전부 FD_26에서).

import os
import sys
import requests
from dotenv import load_dotenv
from supabase import create_client

load_dotenv("C:\Projects\FD_26\.env")
OPENDART_KEY = os.getenv("OPENDART_API_KEY")
SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_PUBLISHABLE_KEY = os.getenv("SUPABASE_PUBLISHABLE_KEY")
if not OPENDART_KEY or not SUPABASE_URL or not SUPABASE_PUBLISHABLE_KEY:
    raise SystemExit("OPENDART_API_KEY, SUPABASE_URL, SUPABASE_PUBLISHABLE_KEY를 FD_26 환경변수 파일에서 확인한다.")
print("OpenDART와 Supabase 환경변수가 준비됐다.")
print(sys.executable)
# 예상 출력: OpenDART와 Supabase 환경변수가 준비됐다.
# 예상 출력: (자기 프로젝트 경로)/FD_26/.venv/bin/python3


## 🔐 OpenDART와 Supabase 준비 확인

이 셀은 키의 **존재 여부만** 확인한다. 값은 화면에 출력하지 않는다. 하나라도 비어 있으면 환경변수 파일을 고친 뒤 다시 실행한다.


In [ ]:
# 📌 §600-1 🔑 코드는 남기고, 키는 환경에 둔다
# 코드를 작성해주세요 👇
# 환경변수 이름과 키 값이 올바른지 확인한다. 값 자체는 출력하지 않는다.
# 예상 출력: OpenDART와 Supabase 환경변수가 준비됐다.


## 📡 OpenDART에서 현재 API 응답을 받는다

[→ §300 전통적 요청](../notes/FD-04-300__opendart-hard-way.md)

아래 요청은 저장된 CSV나 fixture가 아니라 셀 실행 시 OpenDART API를 호출한다. 연결 기준은 `CFS`, 사업연도는 2025년이다.


In [ ]:
# 📌 §600-2 🧾 응답에서 연결 재무제표만 고른다
# 📌 §600-2 📡 OpenDART API 응답 받기
# 코드를 작성해주세요 👇
# 키 값을 출력하지 말고, 실행 시 OpenDART를 직접 호출한다.
# 예상 출력: [live] 000 응답 행 수: API가 반환한 행 수

## 🔎 연결 기준 네 계정을 고른다

응답의 금액은 원 단위 문자열이다. `CFS` 행 중 매출액·영업이익·당기순이익·자본총계를 골라 원 단위 정수로 바꾼다.


In [ ]:
# 📌 §600-3 🧾 응답에서 연결 재무제표만 고른다
# 📌 §600-3 🔎 CFS 네 계정 선택
# 코드를 작성해주세요 👇
# 매출액·영업이익·당기순이익(손실)·자본총계를 CFS 기준으로 고른다.
# 예상 출력: 네 계정의 금액이 원 단위 정수로 표시된다

## 🧮 원 단위를 억원으로 바꿔 한 행을 만든다

원 단위와 억원 단위를 섞지 않도록 각 금액을 1억 원으로 나눈다. 같은 열 이름을 SQLite 실습 및 Supabase 표에서 쓴다.


In [ ]:
# 📌 §600-4 🧾 응답에서 연결 재무제표만 고른다
# 📌 §600-4 🧮 financials 행 만들기
# 코드를 작성해주세요 👇
# 네 계정을 억원으로 바꿔 code, year와 함께 payload에 담는다.
# 예상 출력: code=005930, year=2025, net_income=452068, equity=4363203

## ☁️ Supabase에 적재한다

[→ §600 SDK 경로](../notes/FD-04-600__cloud-supabase.md)

`financials` 표는 §610에서 준비한다. `(code, year)`가 이미 있어도 행을 늘리지 않도록 `upsert`를 실행한다.


In [ ]:
# 📌 §600-5 ☁️ 받은 행을 Supabase에 적재하고 다시 읽는다
# 📌 §600-5 ☁️ Supabase SDK로 직접 적재
# 코드를 작성해주세요 👇
# OpenDART 응답으로 만든 payload를 publishable key로 upsert한다.
# 예상 출력: OpenDART 응답을 Supabase에 적재했다.

## 🔍 클라우드에서 같은 행을 다시 읽는다

행이 저장됐는지, SQLite 경유 없이 OpenDART에서 받은 값이 들어왔는지 확인한다.


In [ ]:
# 📌 §600-6 ☁️ 받은 행을 Supabase에 적재하고 다시 읽는다
# 📌 §600-6 🔍 Supabase에서 확인
# 코드를 작성해주세요 👇
# 삼성전자 2025년 당기순이익과 자본총계를 조회한다.
# 예상 출력: [{'net_income': 452068, 'equity': 4363203}]

## 마무리 확인

[§610](../notes/FD-04-610__supabase-mcp-natural-language.md)에서 자연어 SQL·MCP 경로로 같은 `[live]` 데이터 행을 이미 저장·조회했다. 이 노트북에서는 그 행을 Python SDK로 다시 적재·조회해 두 방법을 비교하고 결과를 확인했다.
